# 02 · Tracking, Undoing Changes & `.gitignore`

**Module 16 – Git & Version Control** · Estimated time: 50 min

> From 📘 *Tracking & Managing Changes*, with live examples. Adds `git revert` (the safe undo for shared history) and how to handle committed secrets.

### 🎯 What you will learn
- Reading `git status` states
- Viewing changes with `git diff` (unstaged, staged, between commits)
- Undoing at every stage: `restore`, `restore --staged`, `commit --amend`, `reset --soft/--mixed/--hard`, `revert`
- Ignoring files with `.gitignore` (Python template!)
- Removing and renaming tracked files: `git rm`, `git rm --cached`, `git mv`

In [1]:
import os, tempfile
PLAYGROUND = tempfile.mkdtemp(prefix="git-playground-")
os.environ.update({
    "PLAYGROUND": PLAYGROUND, "GIT_CONFIG_GLOBAL": os.devnull,
    "GIT_AUTHOR_NAME": "Demo Learner", "GIT_AUTHOR_EMAIL": "learner@example.com",
    "GIT_COMMITTER_NAME": "Demo Learner", "GIT_COMMITTER_EMAIL": "learner@example.com",
    "GIT_PAGER": "cat", "GIT_EDITOR": "true",
})

In [2]:
%%bash
cd "$PLAYGROUND" && git init -q -b main repo && cd repo
printf 'def add(a, b):\n    return a + b\n' > calc.py
echo "# Calculator" > README.md
git add . && git commit -q -m "Initial commit"
git log --oneline

cacfd3d Initial commit


## 1. Status states (📘)

In [3]:
%%bash
cd "$PLAYGROUND/repo"
echo "new" > newfile.txt                                  # untracked
printf 'def add(a, b):\n    return a + b\n\ndef sub(a, b):\n    return a - b\n' > calc.py   # modified
echo "A tiny calculator." >> README.md
git add README.md                                         # staged
git status

On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modifie

d:   README.md

Changes not staged for commit:
  (use "git add <file>..." to update what will be com

mitted)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   calc.py



Untracked files:
  (use "git add <file>..." to include in what will be committed)
	newfile.txt



`git status` tells you: which **branch** you're on, what's **staged** (*Changes to be committed*), what's **modified but not staged**, and what's **untracked**. Read it often!

## 2. Viewing changes: `git diff`

In [4]:
%%bash
cd "$PLAYGROUND/repo"
echo "=== git diff (unstaged changes: working dir vs staging) ==="
git diff
echo "=== git diff --staged (what will go into the next commit) ==="
git diff --staged

=== git diff (unstaged changes: working dir vs staging) ===


diff --git a/calc.py b/calc.py
index 4693ad3..6cddd28 100644
--- a/calc.py
+++ b/calc.py
@@ -1,2 +1,

5 @@
 def add(a, b):
     return a + b
+
+def sub(a, b):
+    return a - b


=== git diff --staged (what will go into the next commit) ===


diff --git a/README.md b/README.md
index 22621e3..6b45dba 100644
--- a/README.md
+++ b/README.md
@@ 

-1 +1,2 @@
 # Calculator
+A tiny calculator.


In [5]:
%%bash
cd "$PLAYGROUND/repo"
git add calc.py && git commit -q -m "Add sub() and README description"
echo "=== git diff HEAD~1 HEAD (between the last two commits) ==="
git diff HEAD~1 HEAD --stat
git log --oneline

=== git diff HEAD~1 HEAD (between the last two commits) ===


 README.md | 1 +
 calc.py   | 3 +++
 2 files changed, 4 insertions(+)


dfe8987 Add sub() and README description


cacfd3d Initial commit


`HEAD` = the current commit; `HEAD~1` = its parent; `HEAD~2` = the grandparent...

## 3. Undoing changes: which command when?

```
                         working dir        staging           last commit         shared history
 discard local edits  →  git restore FILE
 unstage              →                     git restore --staged FILE
 fix the last commit  →                                       git commit --amend
 undo local commits   →                                       git reset [--soft|--mixed|--hard]
 undo a pushed commit →                                                           git revert COMMIT
```

### 3.1 Discard working-directory changes: `git restore`

In [6]:
%%bash
cd "$PLAYGROUND/repo"
echo "oops, garbage" >> calc.py
git status --short
git restore calc.py                 # back to the last committed version (git restore . for all files)
git status --short
tail -2 calc.py

 M calc.py
?? newfile.txt


?? newfile.txt


def sub(a, b):
    return a - b


> ⚠️ **Warning:** `git restore` permanently discards uncommitted changes to that file. There's no undo.

### 3.2 Unstage files (keep the changes): `git restore --staged`

In [7]:
%%bash
cd "$PLAYGROUND/repo"
git add newfile.txt
git status --short                   # A  = staged new file
git restore --staged newfile.txt
git status --short                   # ?? = untracked again, the file is still there

A  newfile.txt


?? newfile.txt


> ⚠️ **Older syntax (📘):** `git reset HEAD file.txt` to unstage, and `git checkout -- file.txt` to discard changes. They still work, but Git 2.23 (2019) introduced the clearer **`git restore`** and **`git switch`** commands, which split up the overloaded `git checkout`.

### 3.3 Amend the last commit

Forgot a file or made a typo in the message? (Only for commits you haven't pushed yet.)

In [8]:
%%bash
cd "$PLAYGROUND/repo"
printf 'def mul(a, b):\n    return a * b\n' >> calc.py
git commit -q -am "Add mul functon"            # typo in message!
echo "forgotten" > forgotten.txt
git add forgotten.txt
git commit -q --amend -m "Add mul() function"  # replaces the last commit
git log --oneline
git show --stat --oneline HEAD | tail -3

015de66 Add mul() function


dfe8987 Add sub() and README description
cacfd3d Initial commit


 calc.py       | 2 ++
 forgotten.txt | 1 +
 2 files changed, 3 insertions(+)


### 3.4 Reset to a previous commit

| Command | Moves HEAD back | Staging area | Working directory |
|---|---|---|---|
| `git reset --soft HEAD~1` | ✅ | changes **kept staged** | kept |
| `git reset HEAD~1` (`--mixed`, default) | ✅ | cleared | changes **kept** (unstaged) |
| `git reset --hard HEAD~1` | ✅ | cleared | ⚠️ **changes deleted** |

In [9]:
%%bash
cd "$PLAYGROUND/repo"
git reset --soft HEAD~1
echo "after --soft:"; git status --short; git log --oneline
git commit -q -m "Add mul() function (recommitted)"
git reset HEAD~1
echo "after --mixed:"; git status --short
git add . ; git commit -q -m "Add mul() function again"
git reset --hard HEAD~1
echo "after --hard:"; git status --short; git log --oneline; grep -c "def" calc.py

after --soft:


M  calc.py
A  forgotten.txt
?? newfile.txt


dfe8987 Add sub() and README description


cacfd3d Initial commit


Unstaged changes after reset:
M	calc.py


...

 M calc.py
?? forgotten.txt
?? newfile.txt


HEAD is now at dfe8987 Add sub() and README description


after --hard:


dfe8987 Add sub() and README description
cacfd3d Initial commit


2


> ⚠️ `--hard` permanently deletes uncommitted work (though `git reflog` can often rescue *commits* you reset away).

### 3.5 ✅ `git revert`: undo safely on shared branches

`reset` **rewrites** history, which is dangerous once commits are pushed and others have pulled them. `revert` creates a **new commit that undoes** an earlier one: history stays intact.

In [10]:
%%bash
cd "$PLAYGROUND/repo"
echo "BROKEN = True" >> calc.py
git commit -q -am "Introduce a bug"
git revert --no-edit HEAD
git log --oneline
tail -1 calc.py

[main 078f485] Revert "Introduce a bug"
 Date: Tue Oct 20 19:46:11 2026 +0530
 1 file changed, 1 del

etion(-)


078f485 Revert "Introduce a bug"


65c75a8 Introduce a bug
dfe8987 Add sub() and README description
cacfd3d Initial commit


    return a - b


## 4. Ignoring files: `.gitignore`

Not every file belongs in Git: passwords and API keys, virtual environments, caches, build outputs, OS junk.

In [11]:
%%bash
cd "$PLAYGROUND/repo"
cat > .gitignore <<'EOF'
# --- Python ---
__pycache__/
*.py[cod]
.venv/
venv/
*.egg-info/
dist/
build/
.pytest_cache/
.ruff_cache/
.mypy_cache/
.ipynb_checkpoints/

# --- secrets ---
.env
*.key
secret.txt

# --- logs & temp ---
*.log
*.tmp

# --- editors / OS ---
.vscode/
.idea/
*.swp
.DS_Store
Thumbs.db

# exception: track this file even though *.log is ignored
!important.log
EOF
mkdir -p __pycache__ .venv && touch __pycache__/calc.cpython-315.pyc .env app.log important.log
git status --short --untracked-files=all

?? .gitignore
?? important.log


Only `.gitignore` and `important.log` show up: everything else is ignored.

| Pattern | Matches |
|---|---|
| `secret.txt` | that file anywhere |
| `*.log` | all files ending in `.log` |
| `build/` | a directory and everything inside |
| `**/*.bak` | `.bak` files in any sub-directory |
| `/todo.txt` | only in the repository root |
| `!important.log` | exception: track it anyway |

Global ignore file for all your repos (e.g. OS/editor files):
```bash
git config --global core.excludesfile ~/.gitignore_global
```
💡 GitHub keeps ready-made templates: https://github.com/github/gitignore (see `Python.gitignore`).

## 5. Removing and renaming tracked files

In [12]:
%%bash
cd "$PLAYGROUND/repo"
git add .gitignore important.log && git commit -q -m "Add .gitignore"
echo "temp" > old_notes.txt && git add old_notes.txt && git commit -q -m "Add notes"

git mv old_notes.txt notes.txt                 # rename/move, staged automatically
git commit -q -m "Rename old_notes.txt to notes.txt"

git rm -q notes.txt                            # delete from repo AND disk
git commit -q -m "Remove notes.txt"
git log --oneline | head -4

7cd0dab Remove notes.txt
5139e69 Rename old_notes.txt to notes.txt
68762b7 Add notes
e701317 Add .gi

tignore


### Stop tracking a file but keep it on disk: `git rm --cached` (📘 practical example)

You accidentally committed a file that should be ignored:

In [13]:
%%bash
cd "$PLAYGROUND/repo"
echo "DB_PASSWORD=hunter2" > config.env
git add config.env && git commit -q -m "Oops: commit config"
git rm --cached -q config.env                  # untrack, but keep the local file
echo "config.env" >> .gitignore
git add .gitignore && git commit -q -m "Stop tracking config.env and ignore it"
ls config.env && git status --short

config.env


> ⚠️ **The secret is still in the history!** Anyone with the repo can run `git show HEAD~1`. If you ever commit a real secret:
> 1. **Rotate it immediately** (change the password / revoke the key); this is the important part
> 2. To scrub it from history, use **`git filter-repo`** (recommended over the old `git filter-branch`) or **BFG Repo-Cleaner**, then force-push and ask collaborators to re-clone
>
> GitHub's *push protection* also scans for many secret types and blocks them.

## 6. Command summary (📘)

| Command | Description |
|---|---|
| `git status` | show working tree status |
| `git diff` / `git diff --staged` | unstaged / staged changes |
| `git restore <file>` | discard working-directory changes |
| `git restore --staged <file>` | unstage |
| `git commit --amend` | rewrite the last (unpushed) commit |
| `git reset --soft HEAD~1` | undo last commit, keep changes staged |
| `git reset HEAD~1` | undo last commit, keep changes unstaged |
| `git reset --hard HEAD~1` | undo last commit, ⚠️ discard changes |
| `git revert <commit>` | ✅ undo a commit with a new commit (safe when shared) |
| `git rm <file>` / `git rm --cached <file>` | remove / stop tracking |
| `git mv <old> <new>` | rename / move |
| `git reflog` | history of where HEAD has been (rescue tool) |

## 🏋️ Practice (📘 exercise)
1. Create a repository with several files; make changes to multiple files.
2. Use `git status` and `git diff` to review changes; stage only some of them.
3. Create a `.gitignore` for a Python project and verify ignored files don't appear in `git status`.
4. Practise `git restore`, `git commit --amend`, and the three `reset` modes (careful with `--hard`!).
5. **Challenge:** commit a fake "secret" file, then stop tracking it, add it to `.gitignore`, and (optionally) remove it from history with `git filter-repo --path secret.txt --invert-paths`.